# Cross-Dataset Classification Matrix

Three output-free, reproducible evaluation blocks share capped parent cohorts.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

from classification_utils import split_cohort
from cross_dataset_classification_utils import (
    DATASETS, EVALUATION_REGIMES, SCORER_LLMS, TRAINING_REGIMES,
    build_deterministic_parent_cohorts, build_shared_parent_manifest, concat_or_empty, evaluate_whole_text_datasets,
    label_result_frame, pivot_matrix, select_scalar_partition, source_coverage_row, train_whole_text_models,
)
from local_evidence_evaluation_utils import (
    combine_local_evidence_dataset_results, evaluate_local_evidence_datasets,
    evaluate_local_evidence_semeval,
)
from local_evidence_training_utils import train_local_evidence_models
from local_evidence_utils import (load_or_build_local_evidence_windows, split_local_evidence_texts)
from whole_text_classification_utils import load_whole_text_cohort

sns.set_theme(style="whitegrid", context="notebook")
DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))
PROCESSED_ROOT = DATA_ROOT / "processed"
DATASETS = {"Kaggle": "kaggle_train", "MAGE": "mage_train", "Outfox": "outfox_chatgpt_train", "BBC": "bbc_news_content"}
LLM_MODELS = ["gemma_3_1b_it", "gemma_3_270m_it", "littlelamb", "smollm2_360m"]
TRAINING_REGIMES = {"Kaggle": ("Kaggle",), "MAGE": ("MAGE",), "Outfox": ("Outfox",), "Mixed": ("Kaggle", "MAGE", "Outfox")}
EVALUATION_REGIMES = {"Kaggle": ("Kaggle",), "MAGE": ("MAGE",), "Outfox": ("Outfox",), "BBC": ("BBC",)}
CLASSIFIER_NAMES = ("logistic_regression",)
WINDOW_LENGTHS = (32, 64, 128)
STRIDE, TOP_K, SEED, MAX_PER_SPLIT = 16, 20, 42, 500
SEMEVAL_DATASETS = ("semeval2024_task8_subtask_c_train", "semeval2024_task8_subtask_c_dev")
SEMEVAL_PREPARED_ROOT = DATA_ROOT / "raw/semevals_2024-task8/SemEval2024-task8/subtaskC"

In [ ]:
source_data, source_coverage = {}, []
for llm in LLM_MODELS:
    source_data[llm] = {}
    for source_name, dataset_alias in DATASETS.items():
        try:
            scalar = load_whole_text_cohort(PROCESSED_ROOT, dataset=dataset_alias, scorer_llm=llm)
            windows = load_or_build_local_evidence_windows(PROCESSED_ROOT, dataset_alias, llm, window_lengths=WINDOW_LENGTHS, stride=STRIDE, top_k=TOP_K)
            candidate_local_partitions = split_local_evidence_texts(windows, seed=SEED, max_per_split=500)
            manifest = build_shared_parent_manifest(scalar, windows, seed=SEED, max_per_split=500)
            local_partitions = {partition: manifest[partition][["split", "text_id", "label"]] for partition in manifest}
            scalar_partitions = {partition: select_scalar_partition(scalar, manifest[partition]) for partition in manifest}
            source_data[llm][source_name] = {"alias": dataset_alias, "scalar": scalar, "windows": windows, "local": local_partitions, "scalar_partitions": scalar_partitions, "manifest": manifest, "candidate_local_partitions": candidate_local_partitions}
            source_coverage.append({"llm": llm, "source_dataset": source_name, "status": "available", "classifier": "source_coverage", "rows": len(scalar)})
        except (FileNotFoundError, KeyError, ValueError, OSError) as error:
            source_coverage.append(source_coverage_row(llm, source_name, error))
            continue

whole_text_metric_frames, whole_text_prediction_frames, whole_text_coverage_frames = [], [], [pd.DataFrame(source_coverage)]
local_evidence_metric_frames, local_evidence_prediction_frames, local_evidence_coverage_frames = [], [], [pd.DataFrame(source_coverage)]
semeval_region_frames, semeval_region_prediction_frames, semeval_boundary_frames, semeval_prediction_frames, semeval_exclusion_frames = [], [], [], [], []
semeval_evaluation_frames = []
local_model_bundles = {}

In [ ]:
# BLOCK 1: WHOLE-TEXT SCALAR METRICS
for llm, datasets in source_data.items():
    for training_regime, training_names in TRAINING_REGIMES.items():
        if not set(training_names).issubset(datasets):
            continue
        train_cohorts = {datasets[name]["alias"]: datasets[name]["scalar_partitions"]["train"] for name in training_names}
        validation_cohorts = {datasets[name]["alias"]: datasets[name]["scalar_partitions"]["validation"] for name in training_names}
        trained = train_whole_text_models(train_cohorts, validation_cohorts, scorer_llm=llm)
        for evaluation_regime, evaluation_names in EVALUATION_REGIMES.items():
            if not set(evaluation_names).issubset(datasets):
                continue
            for partition in ("validation", "test"):
                target = {datasets[name]["alias"]: datasets[name]["scalar_partitions"][partition] for name in evaluation_names}
                evaluated = evaluate_whole_text_datasets(trained.bundle, target, partition=partition)
                for name, frame in (("metrics", evaluated.metrics), ("predictions", evaluated.predictions), ("coverage", evaluated.coverage)):
                    labelled = label_result_frame(frame, llm=llm, training_regime=training_regime, evaluation_regime=evaluation_regime, evaluation_dataset=("Mixed" if evaluation_regime == "Mixed" else ", ".join(evaluation_names)), block="whole_text")
                    if name == "metrics": whole_text_metric_frames.append(labelled)
                    elif name == "predictions": whole_text_prediction_frames.append(labelled)
                    else: whole_text_coverage_frames.append(labelled)

whole_text_metrics = concat_or_empty(whole_text_metric_frames)
whole_text_predictions = concat_or_empty(whole_text_prediction_frames)
whole_text_coverage = concat_or_empty(whole_text_coverage_frames)

In [ ]:
# BLOCK 2: LOCAL EVIDENCE
for llm, datasets in source_data.items():
    for training_regime, training_names in TRAINING_REGIMES.items():
        if not set(training_names).issubset(datasets): continue
        aliases = tuple(datasets[name]["alias"] for name in training_names)
        trained = train_local_evidence_models(PROCESSED_ROOT, aliases, llm, {datasets[name]["alias"]: datasets[name]["local"]["train"] for name in training_names}, {datasets[name]["alias"]: datasets[name]["local"]["validation"] for name in training_names}, classifier_names=CLASSIFIER_NAMES, window_lengths=WINDOW_LENGTHS, stride=STRIDE, top_k=TOP_K, seed=SEED)
        local_model_bundles[(llm, training_regime)] = trained.model
        for evaluation_regime, evaluation_names in EVALUATION_REGIMES.items():
            if not set(evaluation_names).issubset(datasets): continue
            for partition in ("validation", "test"):
                separate_results = [evaluate_local_evidence_datasets(trained.model, PROCESSED_ROOT, (datasets[name]["alias"],), {datasets[name]["alias"]: datasets[name]["local"][partition]}) for name in evaluation_names]
                evaluated = combine_local_evidence_dataset_results(separate_results, evaluation_regime=evaluation_regime) if evaluation_regime == "Mixed" else separate_results[0]
                for name, frame in (("metrics", evaluated.metrics), ("predictions", evaluated.predictions), ("coverage", evaluated.coverage)):
                    labelled = label_result_frame(frame, llm=llm, training_regime=training_regime, evaluation_regime=evaluation_regime, evaluation_dataset=("Mixed" if evaluation_regime == "Mixed" else ", ".join(evaluation_names)), block="local_evidence").assign(partition=partition)
                    if name == "metrics": local_evidence_metric_frames.append(labelled)
                    elif name == "predictions": local_evidence_prediction_frames.append(labelled)
                    else: local_evidence_coverage_frames.append(labelled)

local_evidence_metrics = concat_or_empty(local_evidence_metric_frames)
local_evidence_predictions = concat_or_empty(local_evidence_prediction_frames)
local_evidence_coverage = concat_or_empty(local_evidence_coverage_frames)

In [ ]:
# BLOCK 3: SEMEVAL BOUNDARIES
for llm, datasets in source_data.items():
    for training_regime, training_names in TRAINING_REGIMES.items():
        if not set(training_names).issubset(datasets): continue
        aliases = tuple(datasets[name]["alias"] for name in training_names)
        trained = local_model_bundles[(llm, training_regime)]
        for semeval_dataset in SEMEVAL_DATASETS:
            semeval_path = SEMEVAL_PREPARED_ROOT / ("SemEval2024_task8_C_train_MG.csv" if semeval_dataset.endswith("_train") else "SemEval2024_task8_C_dev_MG.csv")
            try:
                semeval_annotations = pd.read_csv(semeval_path)
                semeval = evaluate_local_evidence_semeval(trained, PROCESSED_ROOT, semeval_annotations, dataset=semeval_dataset, split="MG", sustained_length=3)
                semeval_region_frames.append(label_result_frame(semeval.summary, llm=llm, training_regime=training_regime, evaluation_regime="SemEval", evaluation_dataset=semeval_dataset, block="semeval_region"))
                semeval_region_prediction_frames.append(label_result_frame(semeval.predicted_regions, llm=llm, training_regime=training_regime, evaluation_regime="SemEval", evaluation_dataset=semeval_dataset, block="semeval_region_detail"))
                semeval_evaluation_frames.append(label_result_frame(semeval.evaluations, llm=llm, training_regime=training_regime, evaluation_regime="SemEval", evaluation_dataset=semeval_dataset, block="semeval_region_detail"))
                semeval_boundary_frames.append(label_result_frame(semeval.boundary_summary, llm=llm, training_regime=training_regime, evaluation_regime="SemEval", evaluation_dataset=semeval_dataset, block="semeval_boundary"))
                semeval_prediction_frames.append(label_result_frame(semeval.boundary_predictions, llm=llm, training_regime=training_regime, evaluation_regime="SemEval", evaluation_dataset=semeval_dataset, block="semeval_boundary"))
                semeval_exclusion_frames.append(label_result_frame(semeval.boundary_exclusions, llm=llm, training_regime=training_regime, evaluation_regime="SemEval", evaluation_dataset=semeval_dataset, block="semeval_boundary"))
            except (FileNotFoundError, KeyError, ValueError, OSError) as error:
                semeval_exclusion_frames.append(pd.DataFrame([{"llm": llm, "training_regime": training_regime, "evaluation_dataset": semeval_dataset, "reason": str(error)}]))

semeval_region_summary = concat_or_empty(semeval_region_frames)
semeval_predicted_regions = concat_or_empty(semeval_region_prediction_frames)
semeval_region_evaluations = concat_or_empty(semeval_evaluation_frames)
semeval_boundary_summary = concat_or_empty(semeval_boundary_frames)
boundary_predictions = concat_or_empty(semeval_prediction_frames)
boundary_exclusions = concat_or_empty(semeval_exclusion_frames)

In [ ]:
display(whole_text_coverage, whole_text_metrics, whole_text_predictions, local_evidence_coverage, local_evidence_metrics, local_evidence_predictions, semeval_region_summary, semeval_predicted_regions, semeval_region_evaluations, semeval_boundary_summary, boundary_predictions, boundary_exclusions)
# BBC one-class evaluation is valid for Task 1 metrics; binary ROC-AUC/PR-AUC are not meaningful there.
for block, frame, value, index, columns in (("whole_text", whole_text_metrics, "macro_f1", ["llm", "training_regime", "classifier", "partition", "feature_set"], ["evaluation_regime", "evaluation_dataset"]), ("local_evidence", local_evidence_metrics, "macro_f1", ["llm", "training_regime", "classifier", "partition", "aggregation", "window_length"], ["evaluation_regime", "evaluation_dataset"]), ("semeval_region", semeval_region_summary, "mg_f1", ["llm", "training_regime", "classifier", "window_length"], ["evaluation_dataset"]), ("semeval_boundary", semeval_boundary_summary, "mean_absolute_token_error", ["llm", "training_regime", "classifier", "window_length"], ["evaluation_dataset"])):
    if frame.empty: continue
    matrix = pivot_matrix(frame, value=value, index=index, columns=columns)
    display(matrix)
    sns.heatmap(matrix, annot=True, fmt=".3f", mask=matrix.isna(), vmin=0, vmax=1, cmap="viridis")
    plt.title(f"{block} matrix")
    plt.show()